# Inventory Domain Schema - v2 (Fixed)

This notebook creates the **inventory** schema with the following tables:

| # | Table | Description |
|---|-------|-------------|
| 1 | `inventory.Warehouses` | Warehouse dimension |
| 2 | `inventory.Inventory` | Current inventory levels per product/warehouse |
| 3 | `inventory.InventoryTransactions` | Stock movement transactions |
| 4 | `inventory.PurchaseOrders` | Purchase order headers |
| 5 | `inventory.PurchaseOrderItems` | Purchase order line items |
| 6 | `inventory.DemandForecast` | Demand forecasting data |

### Fixes applied in v2
- `WarehouseID` changed to `INT` (was STRING) for consistency with warehouse dimension
- Renamed `WarehouseLocation` → `WarehouseID` (proper FK reference)
- Removed denormalized columns: `ProductName`, `ProductCategory`, `SupplierName`, `AvailableStock`
- Added `WarehouseID` to `DemandForecast` for warehouse-level forecasting
- Added Primary Key (PK) and Foreign Key (FK) constraints on all tables

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS inventory")

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS inventory.Warehouses (
    WarehouseID INT NOT NULL,
    WarehouseName STRING NOT NULL,
    DisplayName STRING,
    Type STRING,
    Status STRING,
    Location STRING,
    AddressStreet STRING,
    AddressCity STRING,
    AddressState STRING,
    AddressZipCode STRING,
    AddressCountry STRING,
    Phone STRING,
    Email STRING,
    ManagerName STRING,
    ManagerEmail STRING,
    WarehousePriority DECIMAL(3,2),
    MaxCapacity INT,
    OperatingHours STRING,
    StaffCount INT,
    AutomationLevel STRING,
    DeliveryName STRING,
    CreatedBy STRING,
    CreatedDate DATE,
    LastUpdated DATE
) USING DELTA
""")

# spark.sql("ALTER TABLE inventory.Warehouses ADD CONSTRAINT PK_Warehouses PRIMARY KEY (WarehouseID)")

print("✅ inventory.Warehouses created ")

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS inventory.Inventory (
    InventoryID STRING NOT NULL,
    ProductID STRING NOT NULL,
    WarehouseID INT NOT NULL,
    CurrentStock INT,
    ReservedStock INT,
    SafetyStockLevel INT,
    ReorderPoint INT,
    MaxStockLevel INT,
    LastUpdated DATE,
    AverageCost DECIMAL(10,2),
    Status STRING,
    CreatedBy STRING,
    CreatedDate DATE
) USING DELTA
""")

# spark.sql("ALTER TABLE inventory.Inventory ADD CONSTRAINT PK_Inventory PRIMARY KEY (InventoryID)")
# spark.sql("ALTER TABLE inventory.Inventory ADD CONSTRAINT FK_Inventory_Product FOREIGN KEY (ProductID) REFERENCES product.Product(ProductID)")
# spark.sql("ALTER TABLE inventory.Inventory ADD CONSTRAINT FK_Inventory_Warehouse FOREIGN KEY (WarehouseID) REFERENCES inventory.Warehouses(WarehouseID)")

print("✅ inventory.Inventory created ")

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS inventory.InventoryTransactions (
    TransactionID STRING NOT NULL,
    ProductID STRING NOT NULL,
    WarehouseID INT NOT NULL,
    TransactionType STRING,
    TransactionDate DATE,
    Quantity INT,
    UnitCost DECIMAL(10,2),
    TotalValue DECIMAL(10,2),
    ReferenceNumber STRING,
    ReasonCode STRING,
    StockBefore INT,
    StockAfter INT,
    ProcessedBy STRING,
    Notes STRING,
    CreatedBy STRING,
    CreatedDate DATE
) USING DELTA
""")

# spark.sql("ALTER TABLE inventory.InventoryTransactions ADD CONSTRAINT PK_InventoryTransactions PRIMARY KEY (TransactionID)")
# spark.sql("ALTER TABLE inventory.InventoryTransactions ADD CONSTRAINT FK_InvTxn_Product FOREIGN KEY (ProductID) REFERENCES product.Product(ProductID)")
# spark.sql("ALTER TABLE inventory.InventoryTransactions ADD CONSTRAINT FK_InvTxn_Warehouse FOREIGN KEY (WarehouseID) REFERENCES inventory.Warehouses(WarehouseID)")

print("✅ inventory.InventoryTransactions created ")

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS inventory.PurchaseOrders (
    PurchaseOrderID STRING NOT NULL,
    PurchaseOrderNumber STRING NOT NULL,
    SupplierID STRING NOT NULL,
    OrderDate DATE,
    ExpectedDeliveryDate DATE,
    ActualDeliveryDate DATE,
    Status STRING,
    TotalOrderValue DECIMAL(12,2),
    WarehouseID INT,
    OrderedBy STRING,
    OrderPriority STRING,
    Notes STRING,
    CreatedBy STRING,
    CreatedDate DATE
) USING DELTA
""")

# spark.sql("ALTER TABLE inventory.PurchaseOrders ADD CONSTRAINT PK_PurchaseOrders PRIMARY KEY (PurchaseOrderID)")
# spark.sql("ALTER TABLE inventory.PurchaseOrders ADD CONSTRAINT FK_PO_Supplier FOREIGN KEY (SupplierID) REFERENCES supplychain.Suppliers(SupplierID)")
# spark.sql("ALTER TABLE inventory.PurchaseOrders ADD CONSTRAINT FK_PO_Warehouse FOREIGN KEY (WarehouseID) REFERENCES inventory.Warehouses(WarehouseID)")

print("✅ inventory.PurchaseOrders created ")

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS inventory.PurchaseOrderItems (
    PurchaseOrderItemID STRING NOT NULL,
    PurchaseOrderID STRING NOT NULL,
    ProductID STRING NOT NULL,
    QuantityOrdered INT,
    QuantityReceived INT,
    UnitCost DECIMAL(10,2),
    LineTotal DECIMAL(12,2),
    Status STRING,
    ExpectedDate DATE,
    ReceivedDate DATE,
    Notes STRING,
    CreatedBy STRING,
    CreatedDate DATE
) USING DELTA
""")

# spark.sql("ALTER TABLE inventory.PurchaseOrderItems ADD CONSTRAINT PK_PurchaseOrderItems PRIMARY KEY (PurchaseOrderItemID)")
# spark.sql("ALTER TABLE inventory.PurchaseOrderItems ADD CONSTRAINT FK_POItem_PO FOREIGN KEY (PurchaseOrderID) REFERENCES inventory.PurchaseOrders(PurchaseOrderID)")
# spark.sql("ALTER TABLE inventory.PurchaseOrderItems ADD CONSTRAINT FK_POItem_Product FOREIGN KEY (ProductID) REFERENCES product.Product(ProductID)")

print("✅ inventory.PurchaseOrderItems created ")

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS inventory.DemandForecast (
    ForecastID STRING NOT NULL,
    ProductID STRING NOT NULL,
    WarehouseID INT,
    ForecastDate DATE,
    ForecastPeriod STRING,
    PredictedDemand INT,
    ConfidenceLevel DECIMAL(5,2),
    SeasonalMultiplier DECIMAL(5,2),
    TrendDirection STRING,
    BaselineDemand INT,
    MethodUsed STRING,
    ForecastHorizon INT,
    ActualDemand INT,
    AccuracyScore DECIMAL(5,2),
    CreatedBy STRING,
    CreatedDate DATE
) USING DELTA
""")

# spark.sql("ALTER TABLE inventory.DemandForecast ADD CONSTRAINT PK_DemandForecast PRIMARY KEY (ForecastID)")
# spark.sql("ALTER TABLE inventory.DemandForecast ADD CONSTRAINT FK_Forecast_Product FOREIGN KEY (ProductID) REFERENCES product.Product(ProductID)")
# spark.sql("ALTER TABLE inventory.DemandForecast ADD CONSTRAINT FK_Forecast_Warehouse FOREIGN KEY (WarehouseID) REFERENCES inventory.Warehouses(WarehouseID)")

print("✅ inventory.DemandForecast created ")